In [1]:
from merge_tables.db.connection import connect_to_postgres_via_duckdb

duck = connect_to_postgres_via_duckdb()


✓ Successfully connected DuckDB to PostgreSQL database 'medisoft'


In [28]:
duck.sql(
    """
    begin;

    create table pg.easybill.contacts as 
    select * from pg.easybill_2026_06.contacts limit 0;

    insert into pg.easybill.contacts
    with all_docs as (
        select 
            * 
        from read_csv('data/Documents-Export-04_06_2026-10_38_52.csv', types={'Posten: Kostenstelle 2': 'VARCHAR', 'Posten: Fibu Konto': 'VARCHAR', 'Kontakt: Kundennummer': 'VARCHAR'}) 

        union all

        select 
            * 
        from read_csv('data/Documents-Export-04_06_2026-10_39_09.csv', types={'Posten: Kostenstelle 2': 'VARCHAR', 'Posten: Fibu Konto': 'VARCHAR', 'Kontakt: Kundennummer': 'VARCHAR'}) 
    )
    select distinct on ("Kontakt: Kontakt ID") 
        * replace (replace("Kontakt: Kundennummer", ' ', '') as "Kontakt: Kundennummer")
    from (select columns('Kontakt: ') from all_docs);

    commit;
    """
)

In [32]:
duck.sql(
    """
    select * from pg.easybill_2026_06.documents
    """
)

┌──────────────┬──────────────────────────┬───────────────┬─────────────────┬─────────────────┬────────────────────────────┬───────────────────────────────┬───────────────────────┬────────────────────────┬───────────────────────┬───────────────────┬────────────────────────────┬─────────────────────────┬──────────────────┬────────────────────────┬────────────────────────┬───────────────────────────────┬───────────────────────────┬──────────────────────────────────┬──────────────────────┬─────────────────────────────┬───────────────────────┬──────────────────────────────┬─────────────────────────────┬───────────────────────────┬────────────────────────┬──────────────────┬─────────────────────┬─────────────────────────┬────────────────────────────────┬──────────────────────┬───────────────────────────────┬───────────────────────────┬───────────────────────┬─────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [33]:
duck.sql(
    """
    with all_docs as (
        select 
            * 
        from read_csv('data/Documents-Export-04_06_2026-10_38_52.csv', types={'Posten: Kostenstelle 2': 'VARCHAR', 'Posten: Fibu Konto': 'VARCHAR', 'Kontakt: Kundennummer': 'VARCHAR'}) 

        union all

        select 
            * 
        from read_csv('data/Documents-Export-04_06_2026-10_39_09.csv', types={'Posten: Kostenstelle 2': 'VARCHAR', 'Posten: Fibu Konto': 'VARCHAR', 'Kontakt: Kundennummer': 'VARCHAR'}) 
    )
    select * from all_docs
    """
)

┌──────────────┬──────────────────────────┬───────────────┬─────────────────┬─────────────────┬────────────────────────────┬───────────────────────────────┬───────────────────────┬────────────────────────┬───────────────────────┬───────────────────┬────────────────────────────┬─────────────────────────┬──────────────────┬────────────────────────┬────────────────────────┬───────────────────────────────┬───────────────────────────┬──────────────────────────────────┬──────────────────────┬─────────────────────────────┬───────────────────────┬──────────────────────────────┬─────────────────────────────┬───────────────────────────┬────────────────────────┬──────────────────┬─────────────────────┬─────────────────────────┬────────────────────────────────┬──────────────────────┬───────────────────────────────┬───────────────────────────┬───────────────────────┬─────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [57]:
duck.sql("rollback;")

In [58]:
# clear any leftover/aborted transaction from a previous failed run
try:
    duck.sql("rollback")
except Exception:
    pass
# refresh DuckDB's cached Postgres catalog so CREATE picks up the new column types
duck.sql("call pg_clear_cache()")

duck.sql(
    """
    create or replace table pg.easybill.documents as 
    select 
        * replace (
            "Kontakt: Postleitzahl"::varchar as "Kontakt: Postleitzahl",
            "Kontakt (Aktuell): Postleitzahl"::varchar as "Kontakt (Aktuell): Postleitzahl",
            "Posten: Kostenstelle 2"::varchar as "Posten: Kostenstelle 2",
            "Posten: Fibu Konto"::varchar as "Posten: Fibu Konto"
        )
    from pg.easybill_2026_06.documents limit 0;

    insert into pg.easybill.documents by name
    with all_docs as (
        select 
            * 
        from read_csv('data/Documents-Export-04_06_2026-10_38_52.csv', types={'Posten: Kostenstelle 2': 'VARCHAR', 'Posten: Fibu Konto': 'VARCHAR', 'Kontakt: Kundennummer': 'VARCHAR', 'Kontakt: Postleitzahl': 'VARCHAR', 'Kontakt (Aktuell): Postleitzahl': 'VARCHAR'}) 

        union all

        select 
            * 
        from read_csv('data/Documents-Export-04_06_2026-10_39_09.csv', types={'Posten: Kostenstelle 2': 'VARCHAR', 'Posten: Fibu Konto': 'VARCHAR', 'Kontakt: Kundennummer': 'VARCHAR', 'Kontakt: Postleitzahl': 'VARCHAR', 'Kontakt (Aktuell): Postleitzahl': 'VARCHAR'}) 
    )
    select * from all_docs;
    """
)